In [10]:
import pandas as pd

In [11]:
def assign_destination(destination: str) -> int:
    if destination == "Broadcast": 
        return 1
    else:
        return 0


def check_null_value(string_to_check: str) -> str:
    if pd.isna(string_to_check):
        return ""
    else:
        return string_to_check


def assign_device(device: str, source: str) -> str:
    if device in ["Phone", "Phone Scan", "Phone Google"]:
        return "Phone"
    else:
        return device


def encode_company_id(company: str, source: str, service_data: str) -> str:
    # Defined, but not named company identifiers f.e. 0x34f5
    if str(company).startswith("0x") and len(company) == 6:
        return "Unknown"
    # Assign the Galaxy S22 "Anonymous" source the company Samsung
    elif pd.isna(company) and source == "Anonymous":
        return "Samsung Electronics Co. Ltd."
    # Reason?
    elif pd.isna(company) and str(service_data).startswith("4a17235"):
        return "Samsung Electronics Co. Ltd."
    elif pd.isna(company):
        return "Undefined"
    else:
        return company


def extract_time(delta_string: str) -> int:
    if pd.isna(delta_string):
        return 0
    else:
        s = str(delta_string)
        num = "".join(ch for ch in s if ch.isdigit())
        return int(num) if num else 0


def check_company_id_existence(company_id: str) -> int:
    if company_id == "Undefined":
        return 0
    else:
        return 1


def extract_length(entry_to_exctract: str) -> int:
    if pd.isna(entry_to_exctract):
        return 0
    else:
        return len(entry_to_exctract)


def is_adv_channel(channel: int) -> bool:
    if channel in [37, 38, 39]:
        return True
    else:
        return False


def check_entry_existence(entry_to_check: str) -> int:
    if pd.isna(entry_to_check):
        return 0
    else:
        return 1


In [12]:
# Read the input CSV file and rename some columns - robust to both naming schemes (Channel vs Channel_Index, UUID16 vs UUID_16)
def read_data_to_df(file_name: str) -> pd.DataFrame:
    dataset_df = pd.read_csv(file_name, encoding='ISO-8859-1', low_memory=False)
    dataset_df.columns = dataset_df.columns.str.replace("Packet time (start to end)", "packet_start_end")
    dataset_df.columns = dataset_df.columns.str.replace("Delta time (end to start)", "delta_end_start")
    dataset_df.columns = dataset_df.columns.str.replace("Delta time (start to start)", "delta_start_start")
    dataset_df.columns = dataset_df.columns.str.replace(" ", "_")
    dataset_df.columns = dataset_df.columns.str.replace("Test", "Device")
    if "Channel" in dataset_df.columns and "Channel_Index" not in dataset_df.columns:
        dataset_df = dataset_df.rename(columns={"Channel": "Channel_Index"})
    if "UUID16" in dataset_df.columns and "UUID_16" not in dataset_df.columns:
        dataset_df = dataset_df.rename(columns={"UUID16": "UUID_16"})
    if "AD_Type" in dataset_df.columns and "Type" not in dataset_df.columns:
        dataset_df = dataset_df.rename(columns={"AD_Type": "Type"})
    return dataset_df

# --- 10h apartment: stable Buds/AirTags + per-2h-block phone RPA whitelists ---
# Group 1 close (-16.6 phone, -17.5 AirTag, -43.9 Buds), Group 2 far (-48.5 phone, -43.x AirTag, -66.5 Buds)
BUDS_G1 = ["dc:b9:92:eb:f6:5d"]
BUDS_G2 = ["f0:c8:c1:88:8c:53"]
# G1 AirTag rotates d5 -> e3 at ~6-8h block; include both for all files (absent ones are no-ops)
AIRTAG_G1 = ["d5:7a:08:44:31:1d", "e3:aa:7b:69:81:19"]
AIRTAG_G2 = ["cf:14:e2:70:1c:4e"]
# c3:ae:1f:f1:47:40 is persistent interferer -> keep as noise (empty)

PHONE_MAP = {
 "2Groups_3Devices_10h_apartment_Hours0To2.csv": (["67:45:5f:4b:d9:36","5f:82:e8:5c:13:ac","46:61:1d:40:b0:46","66:6c:c0:e1:e4:95","58:fc:b7:c1:b9:24","50:be:2e:7d:30:78","6d:02:cc:a6:da:e7","41:49:6a:92:ec:ec","59:b2:b1:7b:a4:81","6c:ad:29:c6:f2:2a","42:53:bc:31:69:70","72:dd:65:82:ab:81","46:de:cb:1a:29:32","5a:a8:03:51:47:59","48:0f:13:0c:95:58","72:f4:a9:b5:97:50"], ["47:66:fe:37:31:16","77:e8:33:92:ba:92","48:f7:1f:76:78:a0","7b:02:84:9d:61:fd","79:c6:4d:03:31:a3","7f:e6:c5:d1:ac:15","4c:96:e5:93:a9:6b","7c:f2:4a:23:d0:cf","5c:2f:7c:e1:89:dd","5f:28:b1:73:be:67","51:17:48:51:73:83","7a:f3:3b:e4:34:7e","64:28:1f:ca:03:6b","65:ab:b8:14:e2:82","4f:12:cb:6a:d8:4f","6a:ec:de:a1:e9:75"]),
 "2Groups_3Devices_10h_apartment_Hours2To4.csv": (["66:e2:e7:2e:7d:cb","5d:c1:b3:1b:6a:fd","60:1e:85:ca:42:2f","41:0c:1e:d7:ed:5f","4d:24:92:e1:1f:72","5f:b5:ab:5d:1c:85","4e:43:e2:1c:48:fb","46:d9:9b:c2:6c:87","6b:00:8d:83:f6:8d","72:95:48:f5:9d:c0","67:45:5f:4b:d9:36","47:b4:e9:6a:2c:06","47:30:42:bc:4b:33","5c:91:c2:af:45:cb","76:14:b4:87:28:91"], ["49:75:cc:aa:92:28","5a:70:4b:4a:99:d8","5e:2c:a9:76:ee:35","47:a8:5f:d1:b4:1c","7b:95:c3:9d:da:93","71:67:37:c1:fe:f4","44:5c:3a:97:e4:77","75:53:4b:e4:95:01","6f:f5:a0:c9:a4:da","50:17:1c:8a:3b:87","59:a5:6e:a0:b8:05","5f:28:b1:73:be:67","44:d0:69:b7:b4:d6","50:19:4a:8c:d1:c3","40:4f:e7:9b:29:15","6a:00:5e:aa:5c:06"]),
 "2Groups_3Devices_10h_apartment_Hours4To6.csv": (["6f:aa:01:2e:2a:9e","6f:7f:82:7c:bd:1b","58:dc:6b:98:0b:31","70:35:da:dc:2b:7a","7f:3e:e9:72:52:57","5c:d2:b5:b1:61:5e","40:32:f8:9c:8f:2c","74:1f:be:ab:3a:ce","40:97:71:d1:cb:51","73:c3:a2:ee:75:67","4f:44:c1:fe:06:48","5c:5a:1e:ab:6b:51","7c:53:30:04:8f:49","68:27:6b:30:ae:27","7c:8a:67:6b:5c:52"], ["58:65:87:90:2f:72","7d:1a:cb:18:2a:0d","47:46:7a:05:87:af","53:cb:bb:7a:ab:c1","54:5c:50:5e:20:b1","5d:d7:24:e9:0a:80","6d:2e:25:0e:a6:d3","74:bb:04:27:40:cc","58:3a:4f:ef:6d:d2","6a:24:a3:90:d0:56","43:02:42:85:c7:57","4d:2f:18:6f:57:65","5c:9c:50:6b:db:34","71:67:37:c1:fe:f4","41:a5:58:a6:28:7f"]),
 "2Groups_3Devices_10h_apartment_Hours6To8.csv": (["76:7b:28:7a:e3:2e","70:0f:dd:47:df:84","63:85:2f:9e:92:f3","71:f5:88:68:26:6a","45:6e:c7:db:c6:48","77:9f:36:a2:19:46","62:f9:85:0f:51:d5","4f:12:b1:96:bc:ab","60:2a:09:38:d9:9f","4e:d1:de:44:9a:60","4b:d8:2d:ba:0e:58","45:83:47:62:64:8c","40:34:5c:ad:2b:f5","6e:b0:42:2d:ba:bb","58:33:68:50:d5:bb"], ["7b:30:c9:36:d7:2d","4f:77:73:db:ff:14","48:ba:78:89:95:d7","5d:59:03:1a:b3:be","70:87:f9:9e:c4:3f","53:54:b2:e8:e6:3a","46:58:b3:ed:3b:1d","45:e2:bf:cd:7c:7d","41:95:2f:37:9e:36","54:b0:d2:92:2e:08","75:18:ef:c8:b8:3a","66:b4:97:e5:2c:bd","55:78:54:0d:0a:08","6a:ba:ea:d3:52:9d","70:50:ed:14:d2:db"]),
 "2Groups_3Devices_10h_apartment_Hours8To10.csv": (["56:22:46:50:7e:f0","69:bc:73:26:10:c1","77:12:b3:6b:41:27","74:2b:0a:43:fa:3b","63:25:d2:d1:72:66","41:bc:7a:e2:cb:06","60:12:cc:5f:e5:02","5e:e3:42:2c:2a:e4","5f:18:ad:3f:28:8d","7a:44:54:45:d0:b5","72:aa:37:a1:18:aa","4f:f0:92:b9:5e:50","4e:c3:05:0b:cd:b0","4f:2e:db:e1:97:4b","76:1e:34:52:79:68"], ["43:f3:ac:bb:bb:93","5e:99:d9:69:7d:2e","77:0b:32:76:c0:4f","72:2b:b7:02:da:24","68:4c:03:0c:a8:6b","60:76:08:8a:26:69","61:e7:cd:8b:57:db","4d:bd:a4:c9:88:88","7a:5d:76:3a:38:d5","50:29:6f:20:9d:ed","7e:5b:51:46:68:92","5b:4e:88:6b:48:ae","6c:89:49:90:91:18","5d:63:7d:25:6a:05","56:b9:4d:c8:5b:c7"]),
}

def build_source_map(phone_g1, phone_g2):
    m={}
    for s in phone_g1: m[s]=("Phone","Phone Scan",1)
    for s in phone_g2: m[s]=("Phone","Phone Scan",2)
    for s in BUDS_G1: m[s]=("Buds","Buds",1)
    for s in BUDS_G2: m[s]=("Buds","Buds",2)
    for s in AIRTAG_G1: m[s]=("AirTag","AirTag",1)
    for s in AIRTAG_G2: m[s]=("AirTag","AirTag",2)
    return m

def initialize_source_dictionaries(all_unique_sources: list) -> None:
    for source in all_unique_sources:
        if str(source) != "nan":
            source_dictionaries[source] = {}
            source_dictionaries[source]["count"] = 0
            source_dictionaries[source]["malformed_count"] = 0
            source_dictionaries[source]["highest_rssi"] = 0
            source_dictionaries[source]["lowest_rssi"] = -100
            source_dictionaries[source]["average_rssi"] = 0
            source_dictionaries[source]["first_occurence"] = -1
            source_dictionaries[source]["last_occurence"] = 0
            source_dictionaries[source]["device"] = "empty"
            source_dictionaries[source]["sub_device"] = ""
            source_dictionaries[source]["group_id"] = 0


In [13]:
# Assign values for each row with either simple transformations or direct value usage
def fill_labelled_columns(packet_data: tuple, device: str, sub_device: str, group_id: int):

    row_data = {
        "time": packet_data.Time,
        "source": packet_data.Source,
        "destination": packet_data.Destination,
        "is_broadcast": assign_destination(packet_data.Destination),
        "length": packet_data.Length,
        "info": packet_data.Info,
        "rssi": packet_data.RSSI,
        "company_id": encode_company_id(packet_data.Company_ID, packet_data.Source, packet_data.Service_Data),
        "has_company_id": check_company_id_existence(encode_company_id(packet_data.Company_ID, packet_data.Source, packet_data.Service_Data)),
        "channel": packet_data.Channel_Index,
        "is_adv_channel": is_adv_channel(packet_data.Channel_Index),
        "device_name": check_null_value(packet_data.Device_Name),
        "uuid16": check_null_value(packet_data.UUID_16),
        "has_uuid16": check_entry_existence(packet_data.UUID_16),
        "len_uuid16": extract_length(packet_data.UUID_16),
        "uuid128": check_null_value(packet_data.UUID128),
        "has_uuid128": check_entry_existence(packet_data.UUID128),
        "data": check_null_value(packet_data.Data),
        "len_data": extract_length(packet_data.Data),
        "ad_type": check_null_value(packet_data.Type),
        "len_ad_type": extract_length(packet_data.Data),
        "service_data": check_null_value(packet_data.Service_Data),
        "len_service_data": extract_length(packet_data.Service_Data),
        "crc": check_null_value(packet_data.CRC),
        "labelled_device": assign_device(device, packet_data.Source),
        "sublabel_device": sub_device,
        "group_id": group_id,
        "group_label_bin": 1 if group_id != 0 else 0,
        "group_label_multi": group_id,  # 0=noise, 1=G1, 2=G2
        "time_start_end": extract_time(packet_data.packet_start_end),
        "delta_end_start": extract_time(packet_data.delta_end_start),
        "delta_start_start": extract_time(packet_data.delta_start_start)
    }

    # To ensure labeling quality, phone-based labels which do not a valid company_id are labeled empty
    # Valid company_ids include: ["Samsung Electronics Co. Ltd.", "Undefined", "Unknown"]
    if row_data["labelled_device"] in ['Phone'] and row_data["company_id"] not in phone_companies:
        row_data["labelled_device"] = "empty"
        row_data["sublabel_device"] = "empty"
        row_data["group_id"] = 0
        row_data["group_label_bin"] = 0
        row_data["group_label_multi"] = 0

    return row_data


In [14]:
# Main raw dataset input parsing method
def parse_dataframe(input_dataframe: pd.DataFrame):
    nan_discarded_counter = 0
    malformed_discarded_counter = 0
    smart_tag_list = []
    phone_list = []
    buds_list = []

    dict_mean_rssi = (
        pd.to_numeric(
            input_dataframe["RSSI"].astype(str).str.replace("dBm", "", regex=False).str.strip(),
            errors="coerce"
        )
        .groupby(input_dataframe["Source"])
        .mean()
        .to_dict()
    )

    for row in input_dataframe.itertuples(index=False):
        if str(row.Source) == "nan":
            nan_discarded_counter += 1
            continue

        source_dictionaries[row.Source]["count"] += 1

        if "Malformed Packet" in row.Info:
            source_dictionaries[row.Source]["malformed_count"] += 1
            malformed_discarded_counter += 1
            continue

        # Parse RSSI string "-50 dBm" -> int
        try:
            current_rssi = int(str(row.RSSI).replace("dBm","").strip())
        except:
            current_rssi = -100
        if source_dictionaries[row.Source]["first_occurence"] == -1:
            source_dictionaries[row.Source]["first_occurence"] = row.Time
        if row.Time > source_dictionaries[row.Source]["last_occurence"]:
            source_dictionaries[row.Source]["last_occurence"] = row.Time
        if current_rssi > source_dictionaries[row.Source]["lowest_rssi"]:
            source_dictionaries[row.Source]["lowest_rssi"] = current_rssi
        if current_rssi < source_dictionaries[row.Source]["highest_rssi"]:
            source_dictionaries[row.Source]["highest_rssi"] = current_rssi
        if source_dictionaries[row.Source]["average_rssi"] == 0:
            source_dictionaries[row.Source]["average_rssi"] = dict_mean_rssi[row.Source]
            
        # --- Priority 1: Whitelist / ground truth MAP for multi-group ---
        # This overrides heuristics and ensures G1 vs G2 separation even with identical packet content / same RSSI
        if row.Source in SOURCE_TO_GROUP:
            dev, sub, grp = SOURCE_TO_GROUP[row.Source]
            source_dictionaries[row.Source]["device"] = dev
            source_dictionaries[row.Source]["sub_device"] = sub
            source_dictionaries[row.Source]["group_id"] = grp
        else:
            # --- Fallback heuristics for non-target / noise and for completeness ---
            # Smart Tag fallback disabled for 2Groups hybrid dataset (no Smart Tags in experiment)
            # if row.Device_Name == "Smart Tag":
            #     source_dictionaries[row.Source]["device"] = "Smart Tag"
            #     source_dictionaries[row.Source]["sub_device"] = "Smart Tag"
            # if row.UUID_16 == "Samsung Electronics Co., Ltd.,Samsung Electronics Co., Ltd.":
            #     source_dictionaries[row.Source]["device"] = "Smart Tag"
            #     source_dictionaries[row.Source]["sub_device"] = "Smart Tag"
            pass
            # Phone via UUID heuristics
            if row.UUID_16 == "Google LLC" and (row.Length == 63 or row.Length == 42 or row.Length == 120): 
                if source_dictionaries[row.Source]["device"] == "empty":
                    source_dictionaries[row.Source]["device"] = "Phone"
                    source_dictionaries[row.Source]["sub_device"] = "Phone Google"
            if row.Info == "ADV_EXT_IND" and row.Length == 39:
                if source_dictionaries[row.Source]["device"] == "empty":
                    source_dictionaries[row.Source]["device"] = "Phone"
                    source_dictionaries[row.Source]["sub_device"] = "Phone Phone"
            if row.UUID_16 == "Samsung Electronics Co. Ltd." and row.Length == 57: 
                if source_dictionaries[row.Source]["device"] == "empty":
                    source_dictionaries[row.Source]["device"] = "Phone"
                    source_dictionaries[row.Source]["sub_device"] = "Phone Phone"
            if row.Info == "SCAN_REQ" and row.Length == 38 and row.Destination in target_group_rsp_scrs:
                if source_dictionaries[row.Source]["device"] == "empty":
                    source_dictionaries[row.Source]["device"] = "Phone"
                    source_dictionaries[row.Source]["sub_device"] = "Phone Scan"
            # Buds fallback: any Samsung ADV_SCAN_IND L63 with Samsung company is Buds (covers unknown Buds)
            # But whitelist already covers G1/G2 Buds, so here we only label remaining Buds as noise or generic
            if row.Info == "ADV_SCAN_IND" and row.Length == 63 and row.Company_ID == "Samsung Electronics Co. Ltd.":
                if source_dictionaries[row.Source]["device"] == "empty" and row.Source not in ALL_BUDS:
                    # keep as empty to avoid mislabeling interferers; or label generic Buds with group 0
                    pass
            # AirTag fallback: Apple ADV_IND L63 (all AirTags) - whitelist covers G1/G2, rest treated as generic AirTag noise
            if row.Company_ID == "Apple, Inc." and row.Info == "ADV_IND" and row.Length == 63:
                if source_dictionaries[row.Source]["device"] == "empty" and row.Source not in ALL_AIRTAGS:
                    # Keep as empty/noise for modeling; if you want to keep as noise AirTag, uncomment:
                    # source_dictionaries[row.Source]["device"] = "AirTag"
                    # source_dictionaries[row.Source]["sub_device"] = "AirTag"
                    # source_dictionaries[row.Source]["group_id"] = 0
                    pass

        # After assigning all values, create a transformed row for the final list of rows
        row_data = fill_labelled_columns(row, source_dictionaries[row.Source]["device"], source_dictionaries[row.Source]["sub_device"], source_dictionaries[row.Source]["group_id"])
        final_list.append(row_data)

    return nan_discarded_counter, malformed_discarded_counter


In [15]:
def find_empty_and_malformed_sources():
    only_malformed_sources = []
    empty_sources = []

    for key in source_dictionaries.keys():
        count = source_dictionaries[key]["count"]
        malformed_count = source_dictionaries[key]["malformed_count"]
        device = source_dictionaries[key]["device"]

        if (count - malformed_count) == 0:
            only_malformed_sources.append(key)
        
        if (count - malformed_count) != 0 and device == "empty":
            empty_sources.append(key)

    print(f"Only Malformed Sources: {len(only_malformed_sources)}")
    print(f"Empty Sources: {len(empty_sources)}")

    return only_malformed_sources, empty_sources

def reassign_sources():
    empty_occurrences = 0
    new_final_list = []

    for row_dict in final_list:
        if row_dict["labelled_device"] == "empty" and row_dict["company_id"] in phone_companies:
            current_source = row_dict["source"]
            # Only reassign if source has known device in dictionary (prevents overriding whitelist)
            if source_dictionaries[current_source]["device"] != "empty":
                row_dict["labelled_device"] = source_dictionaries[current_source]["device"]
                row_dict["sublabel_device"] = source_dictionaries[current_source]["sub_device"]
                row_dict["group_id"] = source_dictionaries[current_source]["group_id"]
                row_dict["group_label_bin"] = 1 if source_dictionaries[current_source]["group_id"] !=0 else 0
                row_dict["group_label_multi"] = source_dictionaries[current_source]["group_id"]

        if row_dict["labelled_device"] != "empty":
            new_final_list.append(row_dict)
        else:
            new_final_list.append(row_dict)
            empty_occurrences += 1

    return empty_occurrences, new_final_list


In [16]:
# Create the final dataframe based on the final_list rows
def create_labelled_dataframe(entry_list: list):
    new_df = pd.DataFrame(data=entry_list, columns=list((final_list[0].keys())))        
    return new_df

def write_new_dataframe(file_to_write: str, df_to_write: pd.DataFrame):
    df_to_write.to_csv(file_to_write, encoding='utf-8', index=False)
    print(f"New CSV written as: {file_to_write}")

def info_extractor(df_input: pd.DataFrame, wanted_label: str):
    seen_sources = {}
    for row in df_input.itertuples(index=False):
        if row.source not in seen_sources.keys() and row.sublabel_device == wanted_label:
            seen_sources[row.source] = {}
            seen_sources[row.source]["first"] = row.time
            seen_sources[row.source]["last"] = row.time
            seen_sources[row.source]["count"] = source_dictionaries[row.source]["count"]
            continue
        if row.source in seen_sources.keys() and row.sublabel_device == wanted_label:
            if seen_sources[row.source]["last"] < row.time:
                seen_sources[row.source]["last"] = row.time
    return seen_sources

def assign_majority_label(input_df: pd.DataFrame):
    # Compute the majority label per source for labelled_device, sublabel and group
    majority_labels = input_df.groupby('source')['labelled_device'].agg(lambda x: x.mode()[0])
    majority_sub = input_df.groupby('source')['sublabel_device'].agg(lambda x: x.mode()[0])
    majority_group = input_df.groupby('source')['group_id'].agg(lambda x: x.mode()[0])
    input_df['labelled_device'] = input_df['source'].map(majority_labels)
    input_df['sublabel_device'] = input_df['source'].map(majority_sub)
    input_df['group_id'] = input_df['source'].map(majority_group)
    input_df['group_label_bin'] = (input_df['group_id'] != 0).astype(int)
    input_df['group_label_multi'] = input_df['group_id']
    return input_df


In [17]:
##############################
#        CODE FIELD 0        #
##############################

# Global variables
final_list = []
source_dictionaries = {}
phone_companies = ["Samsung Electronics Co. Ltd.", "Undefined", "Unknown"]
undefined_list = []


In [18]:
##############################
#        CODE FIELD 1 - loop 5x2h blocks        #
##############################

import pathlib
base = pathlib.Path("../../data/2Groups_3Devices_10h_apartment")
files = sorted([f for f in base.glob("2Groups_3Devices_10h_apartment_Hours*.csv") if "labeled" not in f.name])
print([f.name for f in files])

for path_to_file in files:
    # reset per-file globals
    final_list = []
    source_dictionaries = {}
    fname = pathlib.Path(path_to_file).name
    phone_g1, phone_g2 = PHONE_MAP[fname]
    SOURCE_TO_GROUP = build_source_map(phone_g1, phone_g2)
    ALL_BUDS = set(BUDS_G1 + BUDS_G2)
    ALL_AIRTAGS = set(AIRTAG_G1 + AIRTAG_G2)

    raw_dataframe = read_data_to_df(str(path_to_file))
    all_sources = raw_dataframe["Source"].unique()
    target_group_rsp_scrs = raw_dataframe.loc[(raw_dataframe["Info"] == "SCAN_RSP") & (raw_dataframe["Device_Name"] == "Buds FE"), "Source"].unique()
    if len(target_group_rsp_scrs)==0:
        target_group_rsp_scrs = BUDS_G1 + BUDS_G2
    print(f"\\n=== {fname} ===")
    print(f"target_group_rsp_scrs: {list(target_group_rsp_scrs)}")
    initialize_source_dictionaries(all_sources)
    nan_counter, malformed_counter = parse_dataframe(raw_dataframe)
    malformed_list, empty_list = find_empty_and_malformed_sources()
    empty_counter, filtered_list = reassign_sources()
    labelled_dataframe = create_labelled_dataframe(filtered_list)
    relevant_sources = labelled_dataframe["source"].unique()
    print(f"Sources total: {len(all_sources)}, labelled rows: {len(labelled_dataframe)}")

    # CODE FIELD 2 inline (no RSSI filtering, whitelist only)
    whitelisted_sources = set(SOURCE_TO_GROUP.keys())
    print("--------------------------------------------------")
    print("FINAL DATAFRAME (before majority vote)")
    print(f"Length Raw: {len(raw_dataframe)}, Labelled: {len(labelled_dataframe)}, Removed malformed {malformed_counter} nan {nan_counter}")
    print(labelled_dataframe.groupby("labelled_device").agg(packets=("labelled_device", "size"), sources=("source", "nunique")).sort_values(by="packets", ascending=False))
    labelled_dataframe = assign_majority_label(labelled_dataframe)

    # CODE FIELD 3 inline
    print("\\n--- After majority vote ---")
    print(labelled_dataframe.groupby("labelled_device").agg(packets=("labelled_device", "size"), sources=("source", "nunique")).sort_values(by="packets", ascending=False))
    print(labelled_dataframe.groupby("group_id").agg(packets=("group_id", "size"), sources=("source", "nunique")).sort_values(by="group_id"))
    print(labelled_dataframe.groupby(["labelled_device","group_id"]).agg(packets=("labelled_device", "size"), sources=("source", "nunique")))

    # save
    new_file_name = pathlib.Path(fname).stem + "_labeled"
    labelled_dataframe.to_csv(base / f"{new_file_name}.csv", index=False)
    print(f"WROTE {base / (new_file_name+'.csv')}")


['2Groups_3Devices_10h_apartment_Hours0To2.csv', '2Groups_3Devices_10h_apartment_Hours2To4.csv', '2Groups_3Devices_10h_apartment_Hours4To6.csv', '2Groups_3Devices_10h_apartment_Hours6To8.csv', '2Groups_3Devices_10h_apartment_Hours8To10.csv']
\n=== 2Groups_3Devices_10h_apartment_Hours0To2.csv ===
target_group_rsp_scrs: ['f0:c8:c1:88:8c:53', 'dc:b9:92:eb:f6:5d', 'f0:c8:c1:a8:8c:53']
Only Malformed Sources: 7934
Empty Sources: 9646
Sources total: 17693, labelled rows: 457469
--------------------------------------------------
FINAL DATAFRAME (before majority vote)
Length Raw: 494674, Labelled: 457469, Removed malformed 35475 nan 1730
                 packets  sources
labelled_device                  
empty             273875     9646
Buds              133089        2
Phone              33840      108
AirTag             16665        2
\n--- After majority vote ---
                 packets  sources
labelled_device                  
empty             273875     9646
Buds              133089  

In [21]:
def group_by_src(df):
    # Simple robust version - converts RSSI to numeric first
    df = df.copy()
    df["rssi_num"] = pd.to_numeric(df["rssi"].astype(str).str.replace("dBm","",regex=False).str.strip(), errors="coerce")
    # Use simple agg without dict lambda that can fail
    result = (
    df.groupby("source")
      .agg(
          count=("rssi_num", "count"),
          RSSI_min=("rssi_num", "min"),
          RSSI_max=("rssi_num", "max"),
          RSSI_avg=("rssi_num", "mean"),
          label=("labelled_device", lambda x: ", ".join(x.unique())),
          group=("group_id", lambda x: ", ".join(map(str, x.unique())))
      )
      .reset_index()                
      .sort_values(by="count", ascending=False)
      .reset_index(drop=True)
    )
    return result
